# Excited States with SSVQE

Find the first excited state of H2 using Subspace-Search VQE (SSVQE) on the Z2-tapered single-qubit Hamiltonian.

**Objectives:**

- Build the tapered single-qubit H2 Hamiltonian `H = H2_C0*I + H2_CZ*Z + H2_CX*X` and read off both exact eigenvalues in closed form.
- Understand why plain VQE only ever finds the *ground* state, and how SSVQE lifts that limitation.
- Pass two ORTHOGONAL inputs |0> and |1> through a single shared unitary U(theta) = Ry(theta) and minimize a weighted energy sum.
- Recover BOTH the ground and first-excited energies from one optimization and compare against exact diagonalization.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: run this cell first.
# Browser-runnable: executes in-page against qcsim, a pure-NumPy Braket stand-in.

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)  # deterministic: this notebook runs headlessly in CI

device = LocalSimulator()
print("setup ready")

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. The tapered single-qubit H2 Hamiltonian

The full STO-3G H2 Hamiltonian (`H2_TERMS`) acts on four qubits. Exploiting its Z2
symmetries lets us *taper* it down to a single qubit without losing any of the
low-lying spectrum. The chemistry kit gives us the three real coefficients of that
tapered operator:

$$ H = c_0\, I + c_z\, Z + c_x\, X $$

with `c0 = H2_C0`, `cz = H2_CZ`, `cx = H2_CX`. This is a 2x2 real symmetric matrix,
so its two eigenvalues are the ground and first-excited energies of H2 in this
minimal model. We never redefine these coefficients — they come straight from the
verified kit.

In [ ]:
# Build the dense 2x2 tapered Hamiltonian from the injected kit coefficients.
I = np.eye(2)
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
X = np.array([[0.0, 1.0], [1.0, 0.0]])

H_tapered = H2_C0 * I + H2_CZ * Z + H2_CX * X

print("c0 = H2_C0 =", H2_C0)
print("cz = H2_CZ =", H2_CZ)
print("cx = H2_CX =", H2_CX)
print("\nH_tapered =")
print(np.array2string(H_tapered, precision=6, suppress_small=True))

# Sanity-cross-check: kit also exposes hamiltonian_matrix for full Pauli strings.
# Here we built H directly from c0/cz/cx, so confirm it is Hermitian.
assert np.allclose(H_tapered, H_tapered.conj().T), "H must be Hermitian"
print("\nHermitian: ok")

## 2. Exact eigenvalues — two ways

Because `H = c0*I + cz*Z + cx*X`, the traceless part `cz*Z + cx*X` has eigenvalues
`+/- hypot(cz, cx)`. Adding the identity shift `c0` gives the closed form:

$$ E_{\text{ground}} = c_0 - \sqrt{c_z^2 + c_x^2}, \qquad E_{\text{excited}} = c_0 + \sqrt{c_z^2 + c_x^2}. $$

We confirm this against a direct `numpy.linalg.eigh` diagonalization, and check that
the ground value reproduces the kit's full-Hamiltonian FCI energy `H2_FCI`.

In [ ]:
# Closed-form eigenvalues.
gap_half = np.hypot(H2_CZ, H2_CX)
E_ground_exact = H2_C0 - gap_half
E_excited_exact = H2_C0 + gap_half

# Direct diagonalization of the 2x2 matrix.
eigvals, eigvecs = np.linalg.eigh(H_tapered)
E_ground_diag, E_excited_diag = eigvals[0], eigvals[1]

print("closed-form   ground = %.6f   excited = %.6f" % (E_ground_exact, E_excited_exact))
print("eigh          ground = %.6f   excited = %.6f" % (E_ground_diag, E_excited_diag))
print("\nkit H2_FCI (full 4-qubit ground) = %.6f" % H2_FCI)

# The closed form and the numerical diagonalization must agree exactly.
assert np.isclose(E_ground_exact, E_ground_diag, atol=1e-10)
assert np.isclose(E_excited_exact, E_excited_diag, atol=1e-10)
# The tapered ground energy reproduces the full-Hamiltonian FCI energy.
assert np.isclose(E_ground_exact, H2_FCI, atol=1e-6)
print("\nexact eigenvalues verified: ok")

## 3. Why plain VQE cannot see the excited state

VQE minimizes a single expectation value `<psi(theta)|H|psi(theta)>`. By the
variational principle that quantity is bounded below by the *ground* energy, so the
optimizer is pulled straight to the lowest state. There is nothing in the cost
function that knows or cares about the next state up.

**SSVQE** fixes this by optimizing a whole *subspace* at once. Take two orthogonal
input states (here |0> and |1>), send both through the SAME unitary U(theta), and
minimize a *weighted* sum of their energies with unequal weights `w0 > w1`:

$$ L(\theta) = w_0\,\langle 0|U^\dagger H U|0\rangle + w_1\,\langle 1|U^\dagger H U|1\rangle. $$

Because U is unitary it keeps the two images orthogonal, and the unequal weights
break the tie: the larger weight `w0` forces its image toward the ground state,
leaving the orthogonal image to settle on the first excited state. At the optimum,
U maps |0> -> ground eigenvector and |1> -> excited eigenvector, so we read BOTH
energies off at once. For a 2x2 problem a single `Ry(theta)` is a fully general
real orthogonal U, which is all we need here.

In [ ]:
# Two energy evaluators, kept consistent so we can trust the asserts later.

# (a) Pure-NumPy path: apply Ry(theta) to the |0> and |1> column vectors directly.
def ry_matrix(theta):
    """2x2 Ry(theta) gate matrix."""
    c, s = np.cos(theta / 2.0), np.sin(theta / 2.0)
    return np.array([[c, -s], [s, c]])

def energies_numpy(theta):
    """Return (E0, E1): energies of U|0> and U|1> under H_tapered, in NumPy."""
    U = ry_matrix(theta)
    psi0 = U @ np.array([1.0, 0.0])  # U|0>
    psi1 = U @ np.array([0.0, 1.0])  # U|1>
    E0 = float((psi0.conj() @ H_tapered @ psi0).real)
    E1 = float((psi1.conj() @ H_tapered @ psi1).real)
    return E0, E1

# (b) Circuit path: build a 1-qubit Ry circuit and read its state_vector() from qcsim.
#     |0> input is the natural circuit start; for the |1> input we prepend an X.
def energies_circuit(theta):
    """Return (E0, E1) using qcsim state_vector() of a 1-qubit Ry circuit."""
    c0 = Circuit().ry(0, theta)            # U|0>
    c1 = Circuit().x(0).ry(0, theta)       # U|1>  (X first flips |0>->|1>)
    sv0 = np.array(c0.state_vector())      # length-2 big-endian state vector
    sv1 = np.array(c1.state_vector())
    E0 = float((sv0.conj() @ H_tapered @ sv0).real)
    E1 = float((sv1.conj() @ H_tapered @ sv1).real)
    return E0, E1

# The two evaluators must agree for any theta (here a fixed probe value).
probe = 0.9
assert np.allclose(energies_numpy(probe), energies_circuit(probe), atol=1e-9)
print("numpy and circuit evaluators agree:", energies_numpy(probe))

## 4. Run SSVQE: one grid search, both energies

No SciPy is allowed and none is needed: for a single parameter a dense NumPy grid
over `theta` in `[0, 2*pi)` is exhaustive and exact to the grid resolution. We pick
weights `w0 = 2`, `w1 = 1` and minimize `L(theta) = w0*E0 + w1*E1`. At the winning
theta we report `E0` as the recovered ground energy and `E1` as the recovered
excited energy.

In [ ]:
w0, w1 = 2.0, 1.0  # unequal weights: w0 > w1 breaks the degeneracy

thetas = np.linspace(0.0, 2.0 * np.pi, 4001)
losses = np.empty_like(thetas)
E0_track = np.empty_like(thetas)
E1_track = np.empty_like(thetas)

for i, th in enumerate(thetas):
    E0, E1 = energies_numpy(th)
    E0_track[i] = E0
    E1_track[i] = E1
    losses[i] = w0 * E0 + w1 * E1

best = int(np.argmin(losses))
theta_star = thetas[best]
E_ground_ssvqe = E0_track[best]
E_excited_ssvqe = E1_track[best]

print("optimal theta*        = %.6f rad" % theta_star)
print("SSVQE ground  energy  = %.6f   (exact %.6f)" % (E_ground_ssvqe, E_ground_exact))
print("SSVQE excited energy  = %.6f   (exact %.6f)" % (E_excited_ssvqe, E_excited_exact))
print("spectral gap          = %.6f   (exact %.6f)"
      % (E_excited_ssvqe - E_ground_ssvqe, E_excited_exact - E_ground_exact))

In [ ]:
# Tight correctness checks against the exact spectrum.
# Recovered energies match exact diagonalization to within atol 1e-3 ...
assert np.isclose(E_ground_ssvqe, E_ground_exact, atol=1e-3), "ground not recovered"
assert np.isclose(E_excited_ssvqe, E_excited_exact, atol=1e-3), "excited not recovered"
# ... and the excited state is never below the ground state (correct ordering).
assert E_excited_ssvqe >= E_ground_ssvqe, "excited must be >= ground"

# Cross-check the optimum through the independent circuit-based evaluator too.
E0_circ, E1_circ = energies_circuit(theta_star)
assert np.isclose(E0_circ, E_ground_ssvqe, atol=1e-6)
assert np.isclose(E1_circ, E_excited_ssvqe, atol=1e-6)

print("all SSVQE asserts passed: both energies recovered, ordering preserved")

## 5. Visualize the subspace search

The figure below sweeps `theta` and shows the two energy branches `E0(theta)`
(image of |0>) and `E1(theta)` (image of |1>), plus the weighted loss `L(theta)`.
At the optimal `theta*` the lower branch touches the exact ground line and the upper
branch touches the exact excited line — the two inputs have been rotated onto the
two eigenvectors simultaneously.

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))

ax0.plot(thetas, E0_track, label="E0(theta) = <0|U^dag H U|0>", color="#2a6fdb")
ax0.plot(thetas, E1_track, label="E1(theta) = <1|U^dag H U|1>", color="#d1495b")
ax0.axhline(E_ground_exact, ls="--", color="#2a6fdb", alpha=0.5, label="exact ground")
ax0.axhline(E_excited_exact, ls="--", color="#d1495b", alpha=0.5, label="exact excited")
ax0.axvline(theta_star, ls=":", color="#444", label="theta*")
ax0.set_xlabel("theta (rad)")
ax0.set_ylabel("energy (Hartree)")
ax0.set_title("Energy branches of the two subspace inputs")
ax0.legend(fontsize=8, loc="center right")

ax1.plot(thetas, losses, color="#2e7d32", label="L = w0*E0 + w1*E1")
ax1.axvline(theta_star, ls=":", color="#444", label="theta*")
ax1.set_xlabel("theta (rad)")
ax1.set_ylabel("weighted loss")
ax1.set_title("SSVQE objective (w0=2, w1=1)")
ax1.legend(fontsize=9)

fig.tight_layout()
plt.show()

## Exercises

Work through these by uncommenting and completing the scaffolds. Keep everything in
NumPy / qcsim-safe Braket — no SciPy, no PennyLane.

In [ ]:
# Exercise 1: Equal weights break SSVQE.
# Set w0 == w1 and re-run the grid search. The optimizer can no longer tell the two
# images apart, so it does not pin them to the ground/excited eigenvectors.
# Confirm that the recovered "excited" energy is no longer close to E_excited_exact.
#
# TODO:
# w0_eq, w1_eq = 1.0, 1.0
# loss_eq = np.array([w0_eq*e0 + w1_eq*e1 for e0, e1 in (energies_numpy(t) for t in thetas)])
# b = int(np.argmin(loss_eq))
# print(energies_numpy(thetas[b]))

# Exercise 2: Coordinate / gradient refinement.
# Replace the dense grid with a small parameter-shift gradient descent on theta:
#   dL/dtheta = (L(theta + pi/2) - L(theta - pi/2)) / 2   (parameter-shift rule)
# Start from a seeded random theta (np.random.seed already set) and verify you reach
# the same theta* to atol 1e-3.
#
# TODO:
# def loss(t):
#     e0, e1 = energies_numpy(t); return w0*e0 + w1*e1
# theta = np.random.uniform(0, 2*np.pi)
# for _ in range(200):
#     grad = (loss(theta + np.pi/2) - loss(theta - np.pi/2)) / 2.0
#     theta -= 0.3 * grad
# print(theta, energies_numpy(theta))

# Exercise 3: Three-state SSVQE (conceptual).
# For a 2x2 H there are only two states, but write down what would change for a 4x4
# tapered Hamiltonian: you would pass |00>,|01>,|10> through a 2-qubit U(theta) with
# weights w0 > w1 > w2 to recover the three lowest energies. Sketch the loss.
pass

## Summary

- The Z2-tapered H2 Hamiltonian `H = H2_C0*I + H2_CZ*Z + H2_CX*X` is a 2x2 operator
  whose eigenvalues are `H2_C0 +/- hypot(H2_CZ, H2_CX)` — the ground and first
  excited energies, with the ground value matching the full-Hamiltonian `H2_FCI`.
- Plain VQE only finds the ground state because it minimizes a single expectation
  value bounded below by the ground energy.
- **SSVQE** sends two orthogonal inputs |0> and |1> through one shared unitary
  U(theta) = Ry(theta) and minimizes the weighted sum `w0*E0 + w1*E1` with `w0 > w1`.
  Unitarity keeps the images orthogonal and the unequal weights pin them onto the
  ground and excited eigenvectors, so a single optimization yields BOTH energies.
- We recovered both energies to better than 1e-3 against exact diagonalization, with
  `E_excited >= E_ground` ordering preserved, cross-checked via both a pure-NumPy 2x2
  evaluator and a qcsim `Circuit.state_vector()` evaluator.

**Next:** [`08-hybrid-chemistry-job.ipynb`](08-hybrid-chemistry-job.ipynb) — package a
variational chemistry workload as a production hybrid job.